In [1]:
import pandas as pd 
import numpy as np
import torch 
import torch.nn as nn
from torch.utils.data import Dataset , DataLoader
from torchmetrics import Accuracy, F1Score, AUROC

In [2]:
data = pd.read_csv('dataset_pitt_cookie.csv')
data = data[data['dx_label'].isin(['ProbableAD','Control'])].reset_index(drop=True)

FEATURE_COLS = [
    'n_filled_pauses', 'n_phon_fragments', 'n_paralinguistic',
    'n_retracings', 'n_unintelligible', 'n_pauses',
    'entryage', 'sex', 'educ'
]


print('Valeurs manquantes par feature:')
print(data[FEATURE_COLS].isna().sum())


for col in FEATURE_COLS:
    if data[col].isna().any():
        data[col] = data[col].fillna(data[col].median())

data[FEATURE_COLS].describe()

Valeurs manquantes par feature:
n_filled_pauses     0
n_phon_fragments    0
n_paralinguistic    0
n_retracings        0
n_unintelligible    0
n_pauses            0
entryage            0
sex                 0
educ                0
dtype: int64


,n_filled_pauses,n_phon_fragments,n_paralinguistic,n_retracings,n_unintelligible,n_pauses,entryage,sex,educ
count,469.000000,469.000000,469.000000,469.000000,469.000000,469.000000,469.000000,469.000000,469.000000
mean,3.390192,1.185501,0.584222,3.036247,0.266525,1.328358,67.106610,0.366738,13.345416
std,3.751238,1.848343,1.062093,3.719429,0.753460,1.630002,8.657869,0.482429,2.843512
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,46.000000,0.000000,6.000000
25%,1.000000,0.000000,0.000000,1.000000,0.000000,0.000000,62.000000,0.000000,12.000000
50%,2.000000,1.000000,0.000000,2.000000,0.000000,1.000000,67.000000,0.000000,13.000000
75%,5.000000,2.000000,1.000000,4.000000,0.000000,2.000000,73.000000,1.000000,15.000000
max,36.000000,16.000000,8.000000,31.000000,6.000000,11.000000,88.000000,1.000000,21.000000


In [3]:
from sklearn.model_selection import train_test_split 

id_sujets = data['subject_id'].unique()
id_tr_vl, id_test = train_test_split(id_sujets, test_size=0.15, random_state=40) 
id_tr, id_vl = train_test_split(id_tr_vl, test_size=0.15, random_state=40) 

data_train = data[data['subject_id'].isin(id_tr)].reset_index(drop=True)
data_val   = data[data['subject_id'].isin(id_vl)].reset_index(drop=True)
data_test  = data[data['subject_id'].isin(id_test)].reset_index(drop=True)

print(data_train.shape, data_val.shape, data_test.shape)

(355, 21) (53, 21) (61, 21)


In [4]:

from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
train_features = scaler.fit_transform(data_train[FEATURE_COLS].values)
val_features   = scaler.transform(data_val[FEATURE_COLS].values)
test_features  = scaler.transform(data_test[FEATURE_COLS].values)

print(f'Features shape - Train: {train_features.shape}, Val: {val_features.shape}, Test: {test_features.shape}')
print(f'\nFeatures normalisées (moyenne ≈ 0, std ≈ 1 sur train):')
print(f'Mean: {train_features.mean(axis=0).round(2)}')
print(f'Std:  {train_features.std(axis=0).round(2)}')

Features shape - Train: (355, 9), Val: (53, 9), Test: (61, 9)

Features normalisées (moyenne ≈ 0, std ≈ 1 sur train):
Mean: [-0. -0. -0.  0.  0. -0.  0.  0.  0.]
Std:  [1. 1. 1. 1. 1. 1. 1. 1. 1.]


In [5]:
text_tr = data_train['transcript'].tolist()
tr_labels = (data_train['dx_label']=='ProbableAD').astype(int).tolist()
text_vl = data_val['transcript'].tolist()
val_labels = (data_val['dx_label']=='ProbableAD').astype(int).tolist()
text_test = data_test['transcript'].tolist()
test_labels = (data_test['dx_label']=='ProbableAD').astype(int).tolist()

In [6]:
from transformers import RobertaTokenizer
tokenizer = RobertaTokenizer.from_pretrained('roberta-base')

In [7]:
class nlp_dataset_hybrid(Dataset):
    """Dataset qui retourne le texte + features cliniques."""
    def __init__(self, text, labels, features, tokenizer):
        self.text = text 
        self.labels = labels 
        self.features = features 
        self.tokenizer = tokenizer 
    
    def __len__(self):
        return len(self.labels)
    
    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.text[idx], 
            max_length=256, truncation=True, 
            padding='max_length', return_tensors='pt'
        )
        return {
            'input_ids':      encoding['input_ids'].squeeze(0),
            'attention_mask': encoding['attention_mask'].squeeze(0),
            'features':       torch.tensor(self.features[idx], dtype=torch.float32),
            'labels':         torch.tensor(self.labels[idx], dtype=torch.long)
        }

In [8]:
tr_nlp_dataset   = nlp_dataset_hybrid(text_tr,   tr_labels,   train_features, tokenizer)
val_nlp_dataset  = nlp_dataset_hybrid(text_vl,   val_labels,  val_features,   tokenizer)
test_nlp_dataset = nlp_dataset_hybrid(text_test, test_labels, test_features,  tokenizer)

tr_loader   = DataLoader(tr_nlp_dataset,   shuffle=True, batch_size=8)
val_loader  = DataLoader(val_nlp_dataset,  batch_size=8)
test_loader = DataLoader(test_nlp_dataset, batch_size=8)


batch = next(iter(tr_loader))
print(f"input_ids:      {batch['input_ids'].shape}")
print(f"attention_mask: {batch['attention_mask'].shape}")
print(f"features:       {batch['features'].shape}  ← {len(FEATURE_COLS)} features")
print(f"labels:         {batch['labels'].shape}")

input_ids:      torch.Size([8, 256])
attention_mask: torch.Size([8, 256])
features:       torch.Size([8, 9])  ← 9 features
labels:         torch.Size([8])


In [9]:
from transformers import RobertaModel

class HybridRoBERTa(nn.Module):
  
    def __init__(self, n_features=9, hidden_size=256, dropout=0.3):
        super().__init__()
        self.roberta = RobertaModel.from_pretrained('roberta-base')
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Sequential(
            nn.Linear(768 + n_features, hidden_size),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_size, 2)
        )
        self.loss_fn = nn.CrossEntropyLoss()
    
    def forward(self, input_ids, attention_mask, features, labels=None):
        
        outputs = self.roberta(input_ids=input_ids, attention_mask=attention_mask)
        cls_embed = outputs.last_hidden_state[:, 0, :]  
        cls_embed = self.dropout(cls_embed)
        
        
        combined = torch.cat([cls_embed, features], dim=1) 
        
        
        logits = self.classifier(combined)
        
        loss = self.loss_fn(logits, labels) if labels is not None else None
        
      
        return type('Output', (), {'logits': logits, 'loss': loss})()


model = HybridRoBERTa(n_features=len(FEATURE_COLS)).to('cuda')


total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
head_params  = sum(p.numel() for p in model.classifier.parameters())
print(f'Total parameters: {total_params:,}')
print(f'Head parameters:  {head_params:,}')

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
pooler.dense.weight             | MISSING    | 
pooler.dense.bias               | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
C:\Users\youne\anaconda3\envs\torch_gpu\lib\site-packages\torch\nn\modules\module.py:1326: UserWarning: expandable_segments not supported on this platform (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\builder\

Total parameters: 124,845,314
Head parameters:  199,682


In [10]:

batch = next(iter(tr_loader))
output = model(
    input_ids=batch['input_ids'].to('cuda'),
    attention_mask=batch['attention_mask'].to('cuda'),
    features=batch['features'].to('cuda'),
    labels=batch['labels'].to('cuda')
)
print(f'Logits shape: {output.logits.shape}')
print(f'Loss: {output.loss.item():.4f}')

Logits shape: torch.Size([8, 2])
Loss: 0.7253


In [11]:
optimizer = torch.optim.AdamW([
    {'params': model.roberta.parameters(),    'lr': 2e-5},
    {'params': model.classifier.parameters(), 'lr': 1e-4}
], weight_decay=0.01)

In [12]:
from transformers import get_linear_schedule_with_warmup

NUM_EPOCHS = 3
num_training_steps = len(tr_loader) * NUM_EPOCHS
print(f'Batches par epoch: {len(tr_loader)}')
print(f'Steps totaux:      {num_training_steps}')

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=int(0.088 * num_training_steps),
    num_training_steps=num_training_steps
)
print(f'Warmup steps:      {int(0.088 * num_training_steps)}')

Batches par epoch: 45
Steps totaux:      135
Warmup steps:      11


In [13]:
def Train(model, train_loader, valid_loader, optimizer, epochs):
    metric_train_acc = Accuracy(task='binary').to('cuda')
    metric_valid_acc = Accuracy(task='binary').to('cuda')
    metric_train_auc = AUROC(task='binary').to('cuda')
    metric_valid_auc = AUROC(task='binary').to('cuda')
    metric_train_f1  = F1Score(task='binary').to('cuda')
    metric_valid_f1  = F1Score(task='binary').to('cuda')
    
    for epoch in range(epochs):
        for m in [metric_train_acc, metric_valid_acc, metric_train_auc,
                  metric_valid_auc, metric_train_f1, metric_valid_f1]:
            m.reset()
        train_loss = 0
        valid_loss = 0
        
        model.train()
        for batch in train_loader:
            optimizer.zero_grad()
            input_ids      = batch['input_ids'].to('cuda')
            attention_mask = batch['attention_mask'].to('cuda')
            features       = batch['features'].to('cuda')      
            labels         = batch['labels'].to('cuda')
            
            output = model(input_ids, attention_mask=attention_mask,
                           features=features, labels=labels)     
            
            logits = output.logits
            preds = torch.argmax(logits, dim=1)
            probs = torch.softmax(logits, dim=1)[:, 1]
            metric_train_acc.update(preds, labels)
            metric_train_auc.update(probs, labels)
            metric_train_f1.update(preds, labels)
            
            loss = output.loss
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            scheduler.step()
            train_loss += loss.item()
        
        model.eval()
        for batch in valid_loader:
            input_ids      = batch['input_ids'].to('cuda')
            attention_mask = batch['attention_mask'].to('cuda')
            features       = batch['features'].to('cuda')
            labels         = batch['labels'].to('cuda')
            with torch.no_grad():
                output = model(input_ids, attention_mask=attention_mask,
                               features=features, labels=labels)
                valid_loss += output.loss.item()
                preds = torch.argmax(output.logits, dim=1)
                probs = torch.softmax(output.logits, dim=1)[:, 1]
                metric_valid_acc.update(preds, labels)
                metric_valid_auc.update(probs, labels)
                metric_valid_f1.update(preds, labels)
        
        print(f'epoch-------------{epoch}-------------')
        print(f'Train | Acc: {metric_train_acc.compute():.2f}  AUC: {metric_train_auc.compute():.2f}  F1: {metric_train_f1.compute():.2f}  Loss: {train_loss/len(train_loader):.4f}')
        print(f'Val   | Acc: {metric_valid_acc.compute():.2f}  AUC: {metric_valid_auc.compute():.2f}  F1: {metric_valid_f1.compute():.2f}  Loss: {valid_loss/len(valid_loader):.4f}')

In [14]:
Train(model, tr_loader, val_loader, optimizer, NUM_EPOCHS)

epoch-------------0-------------
Train | Acc: 0.53  AUC: 0.55  F1: 0.59  Loss: 0.6859
Val   | Acc: 0.57  AUC: 0.60  F1: 0.72  Loss: 0.6750
epoch-------------1-------------
Train | Acc: 0.67  AUC: 0.71  F1: 0.73  Loss: 0.6309
Val   | Acc: 0.68  AUC: 0.83  F1: 0.60  Loss: 0.5880
epoch-------------2-------------
Train | Acc: 0.76  AUC: 0.86  F1: 0.76  Loss: 0.4608
Val   | Acc: 0.83  AUC: 0.86  F1: 0.84  Loss: 0.4854


In [15]:

torch.save(model.state_dict(), 'best_roberta_hybrid_seed40.pt')

In [16]:
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, classification_report, confusion_matrix

model.eval()
all_probs_test = []
all_labels_test = []

with torch.no_grad():
    for batch in test_loader:
        output = model(
            input_ids=batch['input_ids'].to('cuda'),
            attention_mask=batch['attention_mask'].to('cuda'),
            features=batch['features'].to('cuda')
        )
        probs = torch.softmax(output.logits, dim=1)[:, 1]
        all_probs_test.append(probs.cpu())
        all_labels_test.append(batch['labels'])

all_probs_test = torch.cat(all_probs_test).numpy()
all_labels_test = torch.cat(all_labels_test).numpy()


preds = (all_probs_test >= 0.5).astype(int)

print('=' * 55)
print('       RÉSULTATS TEST SET — Modèle Hybride')
print('=' * 55)
print(f'Accuracy : {accuracy_score(all_labels_test, preds):.4f}')
print(f'F1       : {f1_score(all_labels_test, preds):.4f}')
print(f'AUC      : {roc_auc_score(all_labels_test, all_probs_test):.4f}')
print()
print('Confusion matrix [[TN, FP], [FN, TP]]:')
print(confusion_matrix(all_labels_test, preds))
print()
print(classification_report(all_labels_test, preds,
                            target_names=['Control', 'ProbableAD'], digits=4))

       RÉSULTATS TEST SET — Modèle Hybride
Accuracy : 0.8525
F1       : 0.8615
AUC      : 0.9062

Confusion matrix [[TN, FP], [FN, TP]]:
[[24  5]
 [ 4 28]]

              precision    recall  f1-score   support

     Control     0.8571    0.8276    0.8421        29
  ProbableAD     0.8485    0.8750    0.8615        32

    accuracy                         0.8525        61
   macro avg     0.8528    0.8513    0.8518        61
weighted avg     0.8526    0.8525    0.8523        61

